<a href="https://colab.research.google.com/github/mauribo2/Coatings/blob/main/Manual_Coating.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#****** Semi-Automated analysis of coating images ******
#The coating direction must be aligned with the y-axis and the coating height with the z-axis.
#The coated region must be manually chosen
#The reference baselime can be established through one of the following three methods:
#reference = 0: baselime is the average height of the uncoated borders of the image.
#reference = 1: baselime is the fitting plane between the boundaries of the coated region.
#reference = 2: baselime is the fitting polynomial of the uncoated borders of the image.

reference = 1

FILE_PATH = "/content/2mm-05bed-250W+900SS-1.xlsx"    # path to your three-column x y z file

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from sklearn.cluster import DBSCAN
from scipy.interpolate import griddata
import pandas as pd
import os
from skimage.filters import threshold_otsu
from scipy.spatial import ConvexHull
from scipy.spatial import Delaunay
from mpl_toolkits.mplot3d import Axes3D


# Extract the base name of the file without its extension
folder_name = os.path.splitext(os.path.basename(FILE_PATH))[0]
DIRECTORY = folder_name
# Check if the directory already exists
if not os.path.exists(DIRECTORY):
    os.makedirs(DIRECTORY)

### START COMPUTING ####

# 1) Load data
df = pd.read_excel(FILE_PATH)
df.columns = df.columns.str.strip().str.lower()
x = df['x'].values
y = df['y'].values
z = df['z'].values
grid = (x[1]-x[0])*(y[1]-y[0])*1e6
#data = np.loadtxt(FILE_PATH)
#x, y, z = data[:,0], data[:,1], data[:,2]
N = len(z)
ymax = max(y); ymin = min(y) # Fixed: ymin was incorrectly set to min(x)
xmax = max(x); xmin = min(x)
print(f"FILE: {FILE_PATH}")
print(f"{N} data points\n")

# Create a grid over the x-y domain
xi = np.linspace(min(x), xmax, 100)
yi = np.linspace(min(y), ymax, 100)
xi, yi = np.meshgrid(xi, yi)
# Interpolate z values onto the grid
zi = griddata((x, y), z, (xi, yi), method='cubic')

#Perform filtering to determine coating area
otsu_threshold_1 = threshold_otsu(z)
print(f"Otsu's threshold for coating region definition: {otsu_threshold_1}")
print("\n\n**** SELECT LOWEST AND HIGHEST 'y' COORDINATES OF THE COATED REGION FROM THE FIGURE BELOW. THE VALUES ARE INTRODUCED IN THE FOLLOWING CELL ****")

mask_above_threshold = z > otsu_threshold_1
x_filtered = x[mask_above_threshold]
y_filtered = y[mask_above_threshold]
z_filtered = z[mask_above_threshold]
coords = np.column_stack((x_filtered, y_filtered))
# Calculate the convex hull of the filtered coordinates
hull = ConvexHull(coords)
# Create a Delaunay triangulation from the hull vertices
tri = Delaunay(hull.points[hull.vertices])
# Identify which of the filtered points lie within the convex hull
points_in_hull = tri.find_simplex(coords) >= 0
# Create x_in_hull and y_in_hull arrays
x_in_hull = x_filtered[points_in_hull]
y_in_hull = y_filtered[points_in_hull]

#Plot selected area in uncorrected image
plt.figure(figsize=(16, 12))
# Dynamically determine vmin and vmax for better visualization
vmin_plot=0.0; vmax_plot=450
levels = np.linspace(vmin_plot, vmax_plot, 10)

contour = plt.contourf(xi, yi, zi, levels=levels, cmap='plasma')
plt.colorbar(contour, label='Height (um)')
plt.xlabel('x (um)')
plt.ylabel('y (um)')
plt.tick_params(axis='y', right=True, labelright=True)
plt.yticks(range(0, int(ymax), 200))
plt.grid(axis = 'y')
plt.show()

print(f"grid size: {grid} mm^2")

In [ ]:
# Define bloudary values
y_low_left = 800
y_low_right = 800
y_high_left = 9400
y_high_right = 9400

import logging
logging.getLogger('matplotlib.font_manager').disabled = True

mask_left = x < xmax/4.0
unique_y_values = np.unique(y[mask_left])
y_left = y[mask_left]
x_left = x[mask_left] # Corrected: should use x for x_left
z_left = z[mask_left] # Filter z for the left region as well

# Calculate the average z for each unique y in the left region
average_z_by_y = []
for y_val in unique_y_values:
    mask_for_y_val = (y_left == y_val)
    avg_z = np.mean(z_left[mask_for_y_val]) # Use the filtered z_left array
    average_z_by_y.append(avg_z)


# Find local minima for overall average_z_by_y (existing logic)
local_minima = []
if len(average_z_by_y) > 1:
    # Check first point
    if average_z_by_y[0] < average_z_by_y[1]:
        local_minima.append({'y': unique_y_values[0], 'z': average_z_by_y[0]})

    # Check points in between
    for i in range(1, len(average_z_by_y) - 1):
        if average_z_by_y[i] < average_z_by_y[i-1] and average_z_by_y[i] < average_z_by_y[i+1]:
            local_minima.append({'y': unique_y_values[i], 'z': average_z_by_y[i]})

    # Check last point
    if len(average_z_by_y) > 1 and average_z_by_y[-1] < average_z_by_y[-2]:
        local_minima.append({'y': unique_y_values[-1], 'z': average_z_by_y[-1]})

if local_minima:
    first_local_min = local_minima[0]
    last_local_min = local_minima[-1]

    print(f"\nFirst local minimum left: y = {first_local_min['y']:.2f} um, z = {first_local_min['z']:.2f} um")
    print(f"Last local minimum left: y = {last_local_min['y']:.2f} um, z = {last_local_min['z']:.2f} um")
    min_y_left = first_local_min['y']
    max_y_left = last_local_min['y']

else:
    print("\nNo local minima found for overall data.")

mask_right = x > xmax /4.0
unique_y_values = np.unique(y[mask_right])
y_right = y[mask_right] # Renamed to y_right for clarity
x_right = x[mask_right] # Corrected: should use x for x_right
z_right = z[mask_right] # Filter z for the right region as well

# Calculate the average z for each unique y in the right region
average_z_by_y = []
for y_val in unique_y_values:
    mask_for_y_val = (y_right == y_val)
    avg_z = np.mean(z_right[mask_for_y_val]) # Use the filtered z_right array
    average_z_by_y.append(avg_z)


# Find local minima for overall average_z_by_y (existing logic)
local_minima = []
if len(average_z_by_y) > 1:
    # Check first point
    if average_z_by_y[0] < average_z_by_y[1]:
        local_minima.append({'y': unique_y_values[0], 'z': average_z_by_y[0]})

    # Check points in between
    for i in range(1, len(average_z_by_y) - 1):
        if average_z_by_y[i] < average_z_by_y[i-1] and average_z_by_y[i] < average_z_by_y[i+1]:
            local_minima.append({'y': unique_y_values[i], 'z': average_z_by_y[i]})

    # Check last point
    if len(average_z_by_y) > 1 and average_z_by_y[-1] < average_z_by_y[-2]:
        local_minima.append({'y': unique_y_values[-1], 'z': average_z_by_y[-1]})

if local_minima:
    first_local_min = local_minima[0]
    last_local_min = local_minima[-1]

    print(f"\nFirst local minimum right: y = {first_local_min['y']:.2f} um, z = {first_local_min['z']:.2f} um")
    print(f"Last local minimum right: y = {last_local_min['y']:.2f} um, z = {last_local_min['z']:.2f} um")
    min_y_right = first_local_min['y']
    max_y_right = last_local_min['y']


else:
    print("\nNo local minima found for overall data.")

In [ ]:
import logging
logging.getLogger('matplotlib.font_manager').disabled = True

mask_x = x <= xmax
y_masked = y[mask_x]
z_masked = z[mask_x]
unique_y_values = np.unique(y_masked)

# Calculate the average z for each unique y (overall)
average_z_by_y = []
for y_val in unique_y_values:
    mask = (y_masked == y_val)
    avg_z = np.mean(z_masked[mask])
    average_z_by_y.append(avg_z)


# Find local minima for overall average_z_by_y (existing logic)
local_minima = []
if len(average_z_by_y) > 1:
    # Check first point
    if average_z_by_y[0] < average_z_by_y[1]:
        local_minima.append({'y': unique_y_values[0], 'z': average_z_by_y[0]})

    # Check points in between
    for i in range(1, len(average_z_by_y) - 1):
        if average_z_by_y[i] < average_z_by_y[i-1] and average_z_by_y[i] < average_z_by_y[i+1]:
            local_minima.append({'y': unique_y_values[i], 'z': average_z_by_y[i]})

    # Check last point
    if len(average_z_by_y) > 1 and average_z_by_y[-1] < average_z_by_y[-2]:
        local_minima.append({'y': unique_y_values[-1], 'z': average_z_by_y[-1]})

if local_minima:
    first_local_min = local_minima[0]
    last_local_min = local_minima[-1]

    print(f"\nFirst local minimum (overall): y = {first_local_min['y']:.2f} um, z = {first_local_min['z']:.2f} um")
    print(f"Last local minimum (overall): y = {last_local_min['y']:.2f} um, z = {last_local_min['z']:.2f} um")
    min_y = first_local_min['y']
    max_y = last_local_min['y']

else:
    print("\nNo local minima found for overall data.")

# Fit cubic equation for regions outside the first and last local minima
if local_minima and len(local_minima) >= 2:
    y_min_fit = first_local_min['y']
    y_max_fit = last_local_min['y']
    mask_fit = (unique_y_values < y_min_fit) | (unique_y_values > y_max_fit)
    y_to_fit = unique_y_values[mask_fit]
    z_to_fit = np.array(average_z_by_y)[mask_fit]
    average_z_above_threshold = np.mean(z_to_fit)
    print(f"Average z above threshold: {average_z_above_threshold:.2f} um")

    if len(y_to_fit) > 3: # Need at least 4 points for a cubic fit
        coeffs = np.polyfit(y_to_fit, z_to_fit, 3)
        poly = np.poly1d(coeffs)

        y_fitted_curve = np.linspace(y_to_fit.min(), y_to_fit.max(), 100)
        z_fitted_curve = poly(y_fitted_curve)

        plt.figure(figsize=(10, 6))
        plt.scatter(unique_y_values, average_z_by_y, s=10)
        #plt.plot(y_fitted_curve, z_fitted_curve, color='red', linestyle='--', linewidth=1, label='Baseline: Cubic Fit - uncoated region')
        plt.axvline(y_min_fit, color='purple', linestyle=':')
        plt.axvline(y_max_fit, color='purple', linestyle=':')

        # Calculate z values at y_min_fit and y_max_fit from the fitted polynomial
        z_at_ymin_fit = first_local_min['z']
        z_at_ymax_fit = last_local_min['z']

        # Calculate slope (m) and intercept (b) of the line between (y_min_fit, z_at_ymin_fit) and (y_max_fit, z_at_ymax_fit)
        m_line = np.nan
        b_line = np.nan
        if y_max_fit != y_min_fit:
            m_line = (z_at_ymax_fit - z_at_ymin_fit) / (y_max_fit - y_min_fit)
            b_line = z_at_ymin_fit - m_line * y_min_fit
            z_plane_min = m_line*ymin + b_line
            z_plane_max = m_line*ymax + b_line

        else:
            print("Warning: y_min_fit and y_max_fit are the same, cannot calculate slope and intercept for the line.")

        # Plot the line between y_min_fit and y_max_fit and their corresponding z values
        #plt.plot([ymin, ymax], [z_plane_min, z_plane_max], color='blue', linestyle='--', linewidth=1, label=f'Baseline: Plane between boundaries')

        #if not np.isnan(average_z_above_threshold):
        #    plt.plot([ymin, ymax], [average_z_above_threshold, average_z_above_threshold], color='orange', linestyle='--', linewidth=1, label=f'Baseline: Average - uncoated region')

        zmax = max(z)
        plt.xlabel('$y (\mu m)$', fontsize=14)
        plt.ylabel(r'$\langle z \rangle_x (\mu m)$', fontsize=14)
        #plt.grid(True)
        #plt.legend()
        #plt.legend(fontsize=16)
        plt.xticks(range(0, int(ymax), 2000))
        plt.yticks(range(0, int(max(average_z_by_y)), 50))
        plt.xticks(fontsize=12) and plt.yticks(fontsize=12)
        plt.xlim(0, ymax)
        plt.savefig(f"{DIRECTORY}/Selecting_Baseline.tiff", dpi=300)
        plt.show()
        print(f"Cubic fit coefficients: a={coeffs[0]:.4e}, b={coeffs[1]:.4e}, c={coeffs[2]:.4e}, d={coeffs[3]:.4e}")
        if not np.isnan(m_line):
            print(f"Linear line between fit points: m={m_line:.4e}, b={b_line:.4e}")
        if not np.isnan(average_z_above_threshold):
            print(f"Average z (below threshold) between local minima: {average_z_above_threshold:.2f} um")
    else:
        print("Not enough data points to perform a cubic fit for the specified regions.")
else:
    print("Not enough local minima found to perform a cubic fit.")

In [ ]:
# Define the two bounding lines (linear interpolation)
y_low_line  = y_low_left  + (y_low_right  - y_low_left)  * (x / xmax)
y_high_line = y_high_left + (y_high_right - y_high_left) * (x / xmax)

# Boolean mask for points inside the band
mask_in = (y >= y_low_line) & (y <= y_high_line)

# Split arrays
x_in,  y_in,  z_in  = x[mask_in],  y[mask_in],  z[mask_in]
x_out, y_out, z_out = x[~mask_in], y[~mask_in], z[~mask_in]

x_in = np.array(x_in); x_out = np.array(x_out)
y_in = np.array(y_in); y_out = np.array(y_out)
z_in = np.array(z_in); z_out = np.array(z_out)

#Plot selected area in uncorrected image
plt.figure(figsize=(16, 12))
# Dynamically determine vmin and vmax for better visualization
vmin_plot=0.0; vmax_plot=450
levels = np.linspace(vmin_plot, vmax_plot, 10)

contour = plt.contourf(xi, yi, zi, levels=levels, cmap='plasma')
cbar = plt.colorbar(contour, label='Height, z($\mu$m)')
cbar.ax.set_ylabel('Height, z($\mu$m)', fontsize=28) # Corrected: Set fontsize on the colorbar's axis label
cbar.ax.tick_params(labelsize=24) # Increased fontsize
plt.xlabel('x ($\mu$m)', fontsize=28) # Increased fontsize
plt.ylabel('y ($\mu$m)', fontsize=28) # Increased fontsize
plt.xticks(fontsize=24) and plt.yticks(fontsize=24)
plt.yticks(range(0, int(ymax), 2000))
plt.plot(x, y_low_line, label="Line Low", color="cyan", linewidth=4)
plt.plot(x, y_high_line, label="Line High", color="cyan",linewidth=4)
#plt.title('Contour plot of uncorrected data', fontsize=20) # Increased fontsize
plt.savefig(f"{DIRECTORY}/Selected_area.tiff", dpi=300)
plt.show()

In [ ]:
#Total Area (mm^2)
Area_tot = (xmax - xmin)*(ymax - ymin)*1e-6
Area_in = Area_tot*len(x_in)/N
Area_out = Area_tot - Area_in

avg_height_out = np.mean(z_out)
avg_height_in = np.mean(z_in)

report = open(f"{DIRECTORY}/report.txt", "w")
report.write(f"Data point in this file:    {N}\n")
report.write(f"Method for defining baseline:\n")
if reference == 0:report.write(f"Average height of uncoated region\n\n")
if reference == 1:report.write(f"Plane between boundaries of coated-uncoated regions\n\n")
if reference == 2:report.write(f"Paraboloid fitting the uncoated region\n\n")
report.write(f"Points inside the selected coating region:   {len(z_in)}\n")
report.write(f"Points outside the selected coating region:   {len(z_out)}\n")
report.write(f"Avg_out: Average height in UNCORRECTED uncoated region:   {avg_height_out:.2f} um\n")
report.write(f"Avg_in: Average height in UNCORRECTED coated region:   {avg_height_in:.2f} um\n")

# Reset lists as they will be repopulated based on reference value (except for reference == 0)
x_fit_flat = []; y_fit_flat = []; z_fit_flat = []; x_plot = []; y_plot = []; z_plot = []

# Determine mask for fitting based on the reference method
if reference == 2:
    # Use points from the uncoated region (x_out, y_out, z_out) that are outside the overall local minima boundaries
    mask_for_fit = ((y_out <= y_min_fit) | (y_out >= y_max_fit))
    x_fit_flat = x_out[mask_for_fit]
    y_fit_flat = y_out[mask_for_fit]
    z_fit_flat = z_out[mask_for_fit]
elif reference == 1:
    # Use points from the uncoated region (x_out, y_out, z_out) within specific bands near the overall local minima
    # This also fixes the ValueError by using '&' and '|' for element-wise boolean operations
    mask_for_fit = ((y_out <= y_min_fit) & (y_out >= y_min_fit - 100)) | \
                   ((y_out >= y_max_fit) & (y_out <= y_max_fit + 100))
    x_fit_flat = x_out[mask_for_fit]
    y_fit_flat = y_out[mask_for_fit]
    z_fit_flat = z_out[mask_for_fit]

mask_for_plot = ((y_out <= y_min_fit) | (y_out >= y_max_fit))
x_plot = x_out[mask_for_plot]
y_plot = y_out[mask_for_plot]
z_plot = z_out[mask_for_plot]

# Convert lists to numpy arrays for plane fitting
x_fit_arr = np.array(x_fit_flat)
y_fit_arr = np.array(y_fit_flat)
z_fit_arr = np.array(z_fit_flat)

if reference == 1:
  # Fit a plane to x_fit_arr, y_fit_arr, z_fit_arr data (z = ax + by + c)
  A = np.c_[x_fit_arr, y_fit_arr, np.ones(len(x_fit_arr))]

# Use least squares to find the coefficients (a, b, c)
  coeffs, residuals, rank, s = np.linalg.lstsq(A, z_fit_arr, rcond=None)
# coeffs, residuals, rank, s = np.linalg.lstsq(A, z_out, rcond=None)
  a, b, c = coeffs

# Use the plane equation to generate z_plane for x, y
  z_plane = a * x + b * y + c

if reference == 2:
    A = np.c_[
        x_fit_arr**2,
        y_fit_arr**2,
        x_fit_arr * y_fit_arr,
        x_fit_arr,
        y_fit_arr,
        np.ones(len(x_fit_arr))  # Corrected from len(x_out) to len(x_fit_arr)
    ]

  # Use the plane equation to generate z_plane for x, y
    coeffs, residuals, rank, s = np.linalg.lstsq(A, z_fit_arr, rcond=None)
    a, b, c, d, e, f = coeffs
    z_plane = (
    a * x**2
    + b * y**2
    + c * x * y
    + d * x
    + e * y
    + f
)
if reference == 0:
  z_plane = np.ones(len(x))*avg_height_out

# Create a meshgrid for the plane based on the range of x_fit and y_fit
# Adjust these ranges if you want the plane to cover a broader area
x_plane = np.linspace(x.min(), x.max(), 10)
y_plane = np.linspace(y.min(), y.max(), 10)
X_plane, Y_plane = np.meshgrid(x_plane, y_plane)
if reference == 2:
  Z_plane = (
      a * X_plane**2
      + b * Y_plane**2
      + c * X_plane * Y_plane
      + d * X_plane
      + e * Y_plane
      + f
  )
if reference == 1:
  Z_plane = a * X_plane + b * Y_plane + c

if reference == 0:
  Z_plane = np.full((len(x_plane), len(y_plane)), avg_height_out)

# Create the 3D plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot the scattered x_fit, y_fit, z_fit points
ax.scatter(x_plot, y_plot, z_plot, c='blue', marker='.', label='uncoated region data', alpha=0.6)

# Plot the fitted plane
ax.plot_surface(X_plane, Y_plane, Z_plane, color='red', alpha=0.3, label='Baseline surface')

# Set labels and title
ax.set_xlabel('X (um)')
ax.set_ylabel('Y (um)')
ax.set_zlabel('Z (um)')
ax.set_title('Fitted Baseline Surface')
ax.legend()

# Manipulate the view angle: (elevation angle in the z plane, azimuth angle in the x-y plane)
ax.view_init(elev=5, azim=5) # You can change these values (e.g., elev=30, azim=45)
if len(z_out) > 0:
    ax.set_zlim(top=1.2*max(z_out))
else:
    ax.set_zlim(top=np.max(z_plot) * 1.2 if len(z_plot) > 0 else 100) # Fallback if z_out is empty

plt.savefig(f"{DIRECTORY}/Baseline_Surface.tiff", dpi=300)
plt.tight_layout()
#plt.show()

#Correct image
z_corrected = z - z_plane
# Interpolate z values onto the grid
zii = griddata((x, y), z_corrected, (xi, yi), method='cubic')

plt.figure(figsize=(16, 12))
vmin=min(z_corrected); vmax=max(z_corrected)
levels = np.arange(-200, 350, 50)
contour = plt.contourf(xi, yi, zii, levels=levels, cmap='plasma')
plt.plot(x, y_low_line, label="Line Low", color="cyan", linewidth=4)
plt.plot(x, y_high_line, label="Line High", color="cyan",linewidth=4)
cbar = plt.colorbar(contour, label='Height, z($\mu$m)')
cbar.ax.set_ylabel('Height, z($\mu$m)', fontsize=28) # Corrected: Set fontsize on the colorbar's axis label
cbar.ax.tick_params(labelsize=24) # Increased fontsize
plt.xlabel('x ($\mu$m)', fontsize=28) # Increased fontsize
plt.ylabel('y ($\mu$m)', fontsize=28) # Increased fontsize
plt.xticks(fontsize=24) and plt.yticks(fontsize=24)
plt.yticks(range(0, int(ymax), 2000))
#plt.grid(axis = 'y')

#plt.title('Corrected Contour Plot')
plt.savefig(f"{DIRECTORY}/Background_Corrected_Image.tiff", dpi=300)
#plt.show()

#Histogram & cumulative distribution of z
n_bins = 100
factor = 1; #z_inc = []; z_plane_inc = []; z_filt = []; z_plane_filt = []

#Correct z_in, z_filtered
if reference == 1:
    z_plane_inc = a * x_in + b * y_in + c
    z_plane_filt = a * x_filtered + b * y_filtered + c
if reference == 2:
  z_plane_inc = (
    a * x_in**2
    + b * y_in**2
    + c * x_in * y_in
    + d * x_in
    + e * y_in
    + f
)
  z_plane_filt = (
    a * x_filtered**2
    + b * y_filtered**2
    + c * x_filtered * y_filtered
    + d * x_filtered
    + e * y_filtered
    + f
)
if reference == 0:
    z_plane_inc = np.ones(len(x_in))*avg_height_out
    z_plane_filt = np.ones(len(x_filtered))*avg_height_out

z_inc = z_in - factor*z_plane_inc
#z_filt = z_filtered - factor*z_plane_filt
z_filt = z_inc
maxfilt = np.max(z_filt)
minfilt = np.min(z_filt)
z_filt = (z_filt-np.min(z_filt))/(maxfilt-minfilt)
#print(maxfilt,np.mean(z_filt),minfilt)

# Apply Otsu's method to the z_filtered values
otsu_threshold = 0
if len(z_filt) > 0 and np.std(z_filt) > 0: # Ensure z_filt is not empty and has variance
    masktest = z_filt > 0
    zmean = np.mean(z_filt[masktest])
    masktest = z_filt > zmean
    if np.any(masktest):
        otsu_threshold = threshold_otsu(z_filt[masktest])
    else:
        print("Warning: No points above mean z_filt for Otsu threshold. Defaulting to 0.")
else:
    print("Warning: z_filt is empty or has no variance. Otsu threshold not applicable. Defaulting to 0.")

#print(otsu_threshold*(maxfilt-minfilt)+minfilt)
#otsu_threshold = min((otsu_threshold*(maxfilt-minfilt)+minfilt)*1.15, 200)
print(f"Otsu's threshold for defining protrusions: {otsu_threshold} um")

# Compute histogram
hist, bins = np.histogram(z_inc, bins=n_bins)
bin_centers = (bins[:-1] + bins[1:]) / 2
# Find the index of the bin with the highest count
mode_index = np.argmax(hist)
mode_bin_center = bin_centers[mode_index]
median = np.median(z_inc)
Q1 = np.percentile(z_inc, 25)
Q3 = np.percentile(z_inc, 75)
# Compute the Interquartile Range
IQR = Q3 - Q1

# Save histogram
np.savetxt(f"{DIRECTORY}/histo.txt",
np.column_stack((bin_centers, hist)),
header="thickness count", comments='')

# Compute & save cumulative distribution
cum_counts = np.cumsum(hist)
cum_dist   = cum_counts / cum_counts[-1]
np.savetxt(f"{DIRECTORY}/cumulative.txt",
          np.column_stack((bin_centers, cum_dist)),
          header="thickness cumulative_prob", comments='')

# Plot histogram
plt.figure(figsize=(6,4))
plt.bar(bin_centers, hist,
            width=(bins[1]-bins[0]),
            color='C0', edgecolor='black')
#plt.xlim(0, 600)
#plt.ylim(0, 35000)
plt.xlabel("Thickness (um)")
plt.ylabel("Count")
#plt.title("Histogram of thicknesses")
plt.tight_layout()
plt.savefig(f"{DIRECTORY}/Thickesses_Histogram.tiff", dpi=300)

x1 = otsu_threshold
x2 = 1.0
plt.axvline(x1, color='red', linestyle='--', linewidth=2, label="Protrusion threshold")
plt.axvline(x2, color='green', linestyle='--', linewidth=2, label="Depression threshold")
plt.legend()
#plt.show()

# Plot cumulative distribution of z
plt.figure(figsize=(6,4))
plt.plot(bin_centers, cum_dist,
            marker='o', color='C1')
plt.axvline(x1, color='red', linestyle='--', label="Protrusion threshold")
plt.axvline(x2, color='green', linestyle='--', label="Depression threshold")
#plt.xlim(0, 550)
plt.ylim(-0.1, 1.1)
plt.xlabel("Thikness (um)")
plt.ylabel("Cumulative Probability")
#plt.title("Cumulative Distribution of thiknesses")
plt.tight_layout()
plt.savefig(f"{DIRECTORY}/Cumulative_Histogram.tiff", dpi=300)
plt.legend()
#plt.show()

report.write("\n*** STATISTICS ***\n")
report.write(f"Otsu: Otsu's threshold for defining protrusions:  {otsu_threshold:.2f} um\n")
report.write(f"Q1: Q1 (25th percentile):  {Q1:.2f} um\n")
report.write(f"Q3: Q3 (75th percentile): {Q3:.2f} um\n")
report.write(f"IQR: Interquartile Range (IQR): {IQR:.2f} um\n")
report.write(f"Mode: {mode_bin_center:.2f}\n")
report.write(f"Median: {median:.2f}\n")

print("\n*** STATISTICS ***")
print(f"Otsu's threshold for defining protrusions:  {otsu_threshold:.2f} um")
print(f"Q1 (25th percentile):  {Q1:.2f} um")
print(f"Q3 (75th percentile): {Q3:.2f} un")
print(f"IQR: Interquartile Range: {IQR:.2f} um")
print(f"Mode: {mode_bin_center:.2f}")
print(f"Median {median:.2f}")

#Determine protrusions
mask_above_threshold = z_inc > otsu_threshold
x_prot = x_in[mask_above_threshold]
y_prot = y_in[mask_above_threshold]
z_prot = z_inc[mask_above_threshold]

#Determine Depressions
mask_dep = z_inc < 0
x_dep = x_in[mask_dep]
y_dep = y_in[mask_dep]
z_dep = z_inc[mask_dep]

# Corrected line for element-wise comparison
mask_norm = (z_inc >= 0) & (z_inc <= otsu_threshold)
x_norm = x_in[mask_norm]
y_norm = y_in[mask_norm]
z_norm = z_inc[mask_norm]

poor_coating = 10
# Corrected line for element-wise comparison
mask_poor = (z_inc >= 0) & (z_inc <= poor_coating)
x_poor = x_in[mask_poor]
y_poor = y_in[mask_poor]
z_poor = z_inc[mask_poor]

plt.figure(figsize=(6,6))
color = "white"
plt.scatter(x_norm, y_norm, c=[color], s=30)
color = "red"
plt.scatter(x_prot, y_prot, c=[color], s=10)
color = "blue"
plt.scatter(x_dep, y_dep, c=[color], s=30)

#plt.xlabel("x")
#plt.ylabel("y")
ax.set_xticks([])
ax.set_yticks([])
plt.savefig(f"{DIRECTORY}/Analyzed_plot.tiff", dpi=300)
#plt.show()

#Calculate relevant areas
N_in = len(x_in)
Atrue = Area_in*len(x_norm)/N_in
Adep = Area_in*len(x_dep)/N_in
Aprot = Area_in*len(x_prot)/N_in
Apoor = Area_in*len(x_poor)/N_in

print(f"Total area: {Area_tot} mm^2")
print(f"Coated region: {Area_in} mm^2, {Area_in/Area_tot*100} %")
print(f"Uncoated region area: {Area_out} mm^2, {Area_out/Area_tot*100} %")
print(f"Coated area (without depression and protrusion): {Atrue} mm^2, {Atrue/Area_in*100} % of coated region")
print(f"Depressed area: {Adep} mm^2, {Adep/Area_in*100} % of coated region")
print(f"protruding area: {Aprot} mm^2, {Aprot/Area_in*100} % of coated region")
print(f"Coated + protruding area: {Atrue + Aprot} mm^2, {(Atrue + Aprot)/Area_in*100} % of coated region")
print("--------------")

gridx = abs(x[1] - x[0])
gridy = abs(y[1] - y[0])
area_el = gridx*gridy

c_vol = 0
for zj in z_norm:
  c_vol += zj*area_el
for zj in z_prot:
  c_vol += zj*area_el

print(f"Coating volume: {c_vol*1e-9} mm^3")

report.write("\n\n*** SURFACE ANALYSIS ***\n")

report.write(f"A_tot: Total image area:  {Area_tot:.2f} mm^2\n")
report.write(f"A_coat: Coating region area:   {Area_in:.2f} mm^2    {Area_in/Area_tot*100:.3f} % of total area\n\n")
report.write(f"A_uncoat: Uncoated region area:  {Area_out:.2f} mm^2   {Area_out/Area_tot*100:.3f} % of total area\n")
report.write(f"A_blue: Uniform coating (without depressions or protrusions):  {Atrue:.2f} mm^2    {Atrue/Area_in*100:.3f} % of coating region\n")
report.write(f"A_depr: Depressed area:    {Adep:.2f} mm^2    {Adep/Area_in*100:.3f} % of coating region\n")
report.write(f"A_prot: protruding area:   {Aprot:.2f} mm^2   {Aprot/Area_in*100:.3f} % of coating region\n")
report.write(f"A_poor: poorly coated area:   {Apoor:.2f} mm^2   {Apoor/Area_in*100:.3f} % of coating region\n")
report.write(f"Coated + protruding area:    {Atrue + Aprot:.2f} mm^2    {(Atrue + Aprot)/Area_in*100:.3f} % of coating region\n")
report.write(f"Coat_vol: Coating volume:    {c_vol*1e-9:.2f} mm^3\n")
report.close()

import shutil
shutil.make_archive(DIRECTORY, "zip", base_dir=DIRECTORY)


In [ ]:
import os

# Define the base content directory where individual experiment folders are located
content_dir = '/content/'
# Define the path for the consolidated results directory
all_results_dir = os.path.join(content_dir, 'All_Results')

# Filter for items that are directories in the current path '.'
folders = [f for f in os.listdir('.') if os.path.isdir(f)]

# Move each individual experiment folder into the 'All_Results' directory
print(f"Moving individual experiment folders to '{all_results_dir}':")
folders_moved_count = 0
for folder_name in folders:
    source_path = os.path.join(content_dir, folder_name)
    destination_path = os.path.join(all_results_dir, folder_name)
    # Check if the source directory exists before trying to move it
    if os.path.isdir(source_path):
        try:
            shutil.move(source_path, destination_path)
            print(f"  Moved: {folder_name}")
            folders_moved_count += 1
        except Exception as e:
            print(f"  Error moving {folder_name}: {e}")
    else:
        print(f"  Warning: Source directory '{source_path}' does not exist, skipping.")

print(f"Finished moving {folders_moved_count} individual experiment folders.")

# Create a zip archive of the 'All_Results' directory
output_zip_path = os.path.join(content_dir, 'All_Results') # Name of the zip file will be All_Results.zip
shutil.make_archive(output_zip_path, 'zip', all_results_dir)
print(f"Successfully created zip archive: {output_zip_path}.zip")

# Remove the original 'All_Results' directory after zipping
#shutil.rmtree(all_results_dir)
print(f"Successfully removed original directory: {all_results_dir}")